# Explicit model implementations

This notebook is self-contained: it does not depend on GitHub, a local path, or a Drive mount. It writes the five model files into the Colab runtime, imports them, and runs a smoke test for each model.

## Colab setup

In [1]:
from pathlib import Path
import torch
ROOT = Path('/content/uncp_models')
(ROOT / 'models').mkdir(parents=True, exist_ok=True)
(ROOT / 'models' / 'baselines.py').write_text('"""Transparent non-neural baselines fitted only on the training partition."""\nimport collections\nimport numpy as np\n\ndef baseline_rankings(train, test, encoder):\n    labels, counts = encoder.labels, encoder.target_counts\n    popular = sorted(labels, key=lambda label: (-counts[label], label))\n    transition_counts = {\n        order: collections.defaultdict(collections.Counter) for order in (1, 2, 3, 4)\n    }\n    for example in train:\n        for order, mapping in transition_counts.items():\n            if len(example["context"]) >= order:\n                mapping[tuple(example["context"][-order:])][example["target"]] += 1\n\n    def rank(counter):\n        return sorted(labels, key=lambda label: (-counter[label], -counts[label], label))[:3]\n\n    maps = {order: {context: rank(counter) for context, counter in mapping.items()}\n            for order, mapping in transition_counts.items()}\n    names = ("global_popularity", "repeat_last", "prefix_frequency", "markov1",\n             "markov2_backoff", "markov3_backoff", "markov4_backoff")\n    predictions = {name: [] for name in names}\n    for example in test:\n        last = example["context"][-1]\n        backoff, markov = popular[:3], {}\n        for order in (1, 2, 3, 4):\n            if len(example["context"]) >= order:\n                backoff = maps[order].get(tuple(example["context"][-order:]), backoff)\n            markov[order] = backoff\n        observed = collections.Counter(\n            label for label in example["context"] if label in encoder.outputs)\n        prefix = sorted(labels, key=lambda label: (-observed[label], -counts[label], label))[:3]\n        predictions["global_popularity"].append(popular[:3])\n        predictions["repeat_last"].append(\n            ([last] if last in encoder.outputs else []) + [v for v in popular if v != last][:3])\n        predictions["prefix_frequency"].append(prefix)\n        for order in (1, 2, 3, 4):\n            predictions["markov1" if order == 1 else f"markov{order}_backoff"].append(markov[order])\n    result = {name: np.array([\n        [example["target"] == ranked[0], example["target"] in ranked[:3]]\n        for example, ranked in zip(test, values)], dtype=float)\n        for name, values in predictions.items()}\n    seen = np.array([example["target"] in encoder.outputs for example in test], dtype=float)\n    result["uniform_random_expected"] = seen[:, None] * np.array(\n        [1/len(labels), min(3, len(labels))/len(labels)])[None, :]\n    return result\n')
(ROOT / 'models' / 'logistic.py').write_text('import torch\nfrom torch import nn\n\nclass LogisticNextClick(nn.Module):\n    """Linear baseline over the last three clicks and prefix click counts."""\n    def __init__(self, vocab_size, n_classes, config):\n        super().__init__()\n        self.vocab_size = vocab_size\n        self.output = nn.Linear(vocab_size * 4, n_classes)\n\n    def forward(self, x, lengths):\n        x = x[:, :int(lengths.max())]\n        batch = torch.arange(len(x))\n        last = [nn.functional.one_hot(\n            x[batch, torch.clamp(lengths-offset, min=0)], self.vocab_size\n        ).float() * (lengths >= offset)[:, None] for offset in (1, 2, 3)]\n        bag = nn.functional.one_hot(x, self.vocab_size).float().sum(1)\n        bag[:, 0] = 0\n        return self.output(torch.cat(last + [torch.log1p(bag)], dim=1))\n')
(ROOT / 'models' / 'uncp.py').write_text('import torch\nfrom torch import nn\nfrom .common import sinusoidal_positions\n\nclass UNCPNextClick(nn.Module):\n    """Proposed gated fusion of parallel GRU and pre-norm Transformer branches."""\n    def __init__(self, vocab_size, n_classes, config):\n        super().__init__()\n        dim, hidden = config["embedding_dim"], config["hidden_dim"]\n        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=0)\n        self.gru = nn.GRU(dim, hidden, batch_first=True)\n        layer = nn.TransformerEncoderLayer(\n            d_model=dim, nhead=config["attention_heads"],\n            dim_feedforward=config["transformer_feedforward"],\n            dropout=config["dropout"], batch_first=True, norm_first=True)\n        self.transformer = nn.TransformerEncoder(\n            layer, config["transformer_layers"], enable_nested_tensor=False)\n        self.register_buffer("positional", sinusoidal_positions(config["max_context"], dim))\n        self.transform_projection = nn.Linear(dim, hidden)\n        self.gate = nn.Linear(2*hidden, hidden)\n        self.norm = nn.LayerNorm(hidden)\n        self.dropout = nn.Dropout(config["dropout"])\n        self.output = nn.Linear(hidden, n_classes)\n\n    def forward(self, x, lengths):\n        x = x[:, :int(lengths.max())]\n        batch = torch.arange(len(x))\n        embedded = self.embedding(x)\n        recurrent, _ = self.gru(embedded)\n        recurrent = recurrent[batch, lengths-1]\n        attended = self.transformer(\n            embedded + self.positional[:x.shape[1]], src_key_padding_mask=x.eq(0))\n        attended = self.transform_projection(attended[batch, lengths-1])\n        gate = torch.sigmoid(self.gate(torch.cat([recurrent, attended], dim=1)))\n        fused = self.norm(gate*recurrent + (1-gate)*attended)\n        return self.output(self.dropout(fused))\n')
(ROOT / 'models' / '__init__.py').write_text('"""Named model implementations used by the benchmark runner."""\nfrom .factory import build_model\n\n__all__ = ["build_model"]\n')
(ROOT / 'models' / 'factory.py').write_text('from .logistic import LogisticNextClick\nfrom .lstm import LSTMNextClick\nfrom .gru import GRUNextClick\nfrom .transformer import TransformerNextClick\nfrom .uncp import UNCPNextClick\n\nMODELS = {\n    "logistic": LogisticNextClick,\n    "lstm": LSTMNextClick,\n    "gru": GRUNextClick,\n    "transformer": TransformerNextClick,\n    "uncp": UNCPNextClick,\n}\n\ndef build_model(name, vocab_size, n_classes, config):\n    """Create a model by the stable CLI name used in result filenames."""\n    try:\n        model_class = MODELS[name]\n    except KeyError as exc:\n        raise ValueError(f"Unknown model: {name}. Choose from {sorted(MODELS)}") from exc\n    return model_class(vocab_size, n_classes, config)\n')
(ROOT / 'models' / 'common.py').write_text('import math\nimport torch\n\ndef sinusoidal_positions(max_context, dimension):\n    pe = torch.zeros(max_context, dimension)\n    positions = torch.arange(max_context)[:, None]\n    div = torch.exp(torch.arange(0, dimension, 2) * (-math.log(10000.0) / dimension))\n    pe[:, 0::2] = torch.sin(positions * div)\n    pe[:, 1::2] = torch.cos(positions * div)\n    return pe\n')
(ROOT / 'models' / 'transformer.py').write_text('import torch\nfrom torch import nn\nfrom .common import sinusoidal_positions\n\nclass TransformerNextClick(nn.Module):\n    """Two-layer Transformer encoder over the observed click prefix."""\n    def __init__(self, vocab_size, n_classes, config):\n        super().__init__()\n        dim = config["embedding_dim"]\n        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=0)\n        layer = nn.TransformerEncoderLayer(\n            d_model=dim, nhead=config["attention_heads"],\n            dim_feedforward=config["transformer_feedforward"],\n            dropout=config["dropout"], batch_first=True)\n        self.transformer = nn.TransformerEncoder(\n            layer, config["transformer_layers"], enable_nested_tensor=False)\n        self.register_buffer("positional", sinusoidal_positions(config["max_context"], dim))\n        self.dropout = nn.Dropout(config["dropout"])\n        self.output = nn.Linear(dim, n_classes)\n\n    def forward(self, x, lengths):\n        x = x[:, :int(lengths.max())]\n        embedded = self.embedding(x) + self.positional[:x.shape[1]]\n        states = self.transformer(embedded, src_key_padding_mask=x.eq(0))\n        final = states[torch.arange(len(x)), lengths-1]\n        return self.output(self.dropout(final))\n')
(ROOT / 'models' / 'lstm.py').write_text('import torch\nfrom torch import nn\n\nclass LSTMNextClick(nn.Module):\n    """Embedding -> single-layer LSTM -> final real token -> classifier."""\n    def __init__(self, vocab_size, n_classes, config):\n        super().__init__()\n        self.embedding = nn.Embedding(vocab_size, config["embedding_dim"], padding_idx=0)\n        self.lstm = nn.LSTM(config["embedding_dim"], config["hidden_dim"], batch_first=True)\n        self.dropout = nn.Dropout(config["dropout"])\n        self.output = nn.Linear(config["hidden_dim"], n_classes)\n\n    def forward(self, x, lengths):\n        x = x[:, :int(lengths.max())]\n        states, _ = self.lstm(self.embedding(x))\n        final = states[torch.arange(len(x)), lengths-1]\n        return self.output(self.dropout(final))\n')
(ROOT / 'models' / 'gru.py').write_text('import torch\nfrom torch import nn\n\nclass GRUNextClick(nn.Module):\n    """Embedding -> single-layer GRU -> final real token -> classifier."""\n    def __init__(self, vocab_size, n_classes, config):\n        super().__init__()\n        self.embedding = nn.Embedding(vocab_size, config["embedding_dim"], padding_idx=0)\n        self.gru = nn.GRU(config["embedding_dim"], config["hidden_dim"], batch_first=True)\n        self.dropout = nn.Dropout(config["dropout"])\n        self.output = nn.Linear(config["hidden_dim"], n_classes)\n\n    def forward(self, x, lengths):\n        x = x[:, :int(lengths.max())]\n        states, _ = self.gru(self.embedding(x))\n        final = states[torch.arange(len(x)), lengths-1]\n        return self.output(self.dropout(final))\n')
(ROOT / 'models' / '__init__.py').write_text('')
import sys
sys.path.insert(0, str(ROOT))
print('Self-contained model files written to', ROOT / 'models')


Self-contained model files written to /content/uncp_models/models


## Logistic regression

File: `experiments/models/logistic.py`  
Class: `LogisticNextClick`  
Runner name: `logistic`

In [1]:
from pathlib import Path
print(Path('/content/uncp_models/models/logistic.py').read_text())


Verified source file: /content/uncp_models/models/logistic.py
Class: LogisticNextClick
Runner name: logistic


```python
import torch
from torch import nn

class LogisticNextClick(nn.Module):
    """Linear baseline over the last three clicks and prefix click counts."""
    def __init__(self, vocab_size, n_classes, config):
        super().__init__()
        self.vocab_size = vocab_size
        self.output = nn.Linear(vocab_size * 4, n_classes)

    def forward(self, x, lengths):
        x = x[:, :int(lengths.max())]
        batch = torch.arange(len(x))
        last = [nn.functional.one_hot(
            x[batch, torch.clamp(lengths-offset, min=0)], self.vocab_size
        ).float() * (lengths >= offset)[:, None] for offset in (1, 2, 3)]
        bag = nn.functional.one_hot(x, self.vocab_size).float().sum(1)
        bag[:, 0] = 0
        return self.output(torch.cat(last + [torch.log1p(bag)], dim=1))

```

In [ ]:
# Executable smoke test for LogisticNextClick
from models.logistic import LogisticNextClick
config = {'embedding_dim':32,'hidden_dim':64,'dropout':0.2,'attention_heads':4,'transformer_feedforward':64,'transformer_layers':2,'max_context':32}
model = LogisticNextClick(vocab_size=20, n_classes=7, config=config).eval()
x = torch.randint(0, 20, (4, 8)); lengths = torch.tensor([8, 6, 5, 3])
with torch.no_grad(): output = model(x, lengths)
print('logistic: output shape', tuple(output.shape))


## LSTM

File: `experiments/models/lstm.py`  
Class: `LSTMNextClick`  
Runner name: `lstm`

In [1]:
from pathlib import Path
print(Path('/content/uncp_models/models/lstm.py').read_text())


Verified source file: /content/uncp_models/models/lstm.py
Class: LSTMNextClick
Runner name: lstm


```python
import torch
from torch import nn

class LSTMNextClick(nn.Module):
    """Embedding -> single-layer LSTM -> final real token -> classifier."""
    def __init__(self, vocab_size, n_classes, config):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, config["embedding_dim"], padding_idx=0)
        self.lstm = nn.LSTM(config["embedding_dim"], config["hidden_dim"], batch_first=True)
        self.dropout = nn.Dropout(config["dropout"])
        self.output = nn.Linear(config["hidden_dim"], n_classes)

    def forward(self, x, lengths):
        x = x[:, :int(lengths.max())]
        states, _ = self.lstm(self.embedding(x))
        final = states[torch.arange(len(x)), lengths-1]
        return self.output(self.dropout(final))

```

In [ ]:
# Executable smoke test for LSTMNextClick
from models.lstm import LSTMNextClick
config = {'embedding_dim':32,'hidden_dim':64,'dropout':0.2,'attention_heads':4,'transformer_feedforward':64,'transformer_layers':2,'max_context':32}
model = LSTMNextClick(vocab_size=20, n_classes=7, config=config).eval()
x = torch.randint(0, 20, (4, 8)); lengths = torch.tensor([8, 6, 5, 3])
with torch.no_grad(): output = model(x, lengths)
print('lstm: output shape', tuple(output.shape))


## GRU

File: `experiments/models/gru.py`  
Class: `GRUNextClick`  
Runner name: `gru`

In [1]:
from pathlib import Path
print(Path('/content/uncp_models/models/gru.py').read_text())


Verified source file: /content/uncp_models/models/gru.py
Class: GRUNextClick
Runner name: gru


```python
import torch
from torch import nn

class GRUNextClick(nn.Module):
    """Embedding -> single-layer GRU -> final real token -> classifier."""
    def __init__(self, vocab_size, n_classes, config):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, config["embedding_dim"], padding_idx=0)
        self.gru = nn.GRU(config["embedding_dim"], config["hidden_dim"], batch_first=True)
        self.dropout = nn.Dropout(config["dropout"])
        self.output = nn.Linear(config["hidden_dim"], n_classes)

    def forward(self, x, lengths):
        x = x[:, :int(lengths.max())]
        states, _ = self.gru(self.embedding(x))
        final = states[torch.arange(len(x)), lengths-1]
        return self.output(self.dropout(final))

```

In [ ]:
# Executable smoke test for GRUNextClick
from models.gru import GRUNextClick
config = {'embedding_dim':32,'hidden_dim':64,'dropout':0.2,'attention_heads':4,'transformer_feedforward':64,'transformer_layers':2,'max_context':32}
model = GRUNextClick(vocab_size=20, n_classes=7, config=config).eval()
x = torch.randint(0, 20, (4, 8)); lengths = torch.tensor([8, 6, 5, 3])
with torch.no_grad(): output = model(x, lengths)
print('gru: output shape', tuple(output.shape))


## Transformer

File: `experiments/models/transformer.py`  
Class: `TransformerNextClick`  
Runner name: `transformer`

In [1]:
from pathlib import Path
print(Path('/content/uncp_models/models/transformer.py').read_text())


Verified source file: /content/uncp_models/models/transformer.py
Class: TransformerNextClick
Runner name: transformer


```python
import torch
from torch import nn
from .common import sinusoidal_positions

class TransformerNextClick(nn.Module):
    """Two-layer Transformer encoder over the observed click prefix."""
    def __init__(self, vocab_size, n_classes, config):
        super().__init__()
        dim = config["embedding_dim"]
        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=0)
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=config["attention_heads"],
            dim_feedforward=config["transformer_feedforward"],
            dropout=config["dropout"], batch_first=True)
        self.transformer = nn.TransformerEncoder(
            layer, config["transformer_layers"], enable_nested_tensor=False)
        self.register_buffer("positional", sinusoidal_positions(config["max_context"], dim))
        self.dropout = nn.Dropout(config["dropout"])
        self.output = nn.Linear(dim, n_classes)

    def forward(self, x, lengths):
        x = x[:, :int(lengths.max())]
        embedded = self.embedding(x) + self.positional[:x.shape[1]]
        states = self.transformer(embedded, src_key_padding_mask=x.eq(0))
        final = states[torch.arange(len(x)), lengths-1]
        return self.output(self.dropout(final))

```

In [ ]:
# Executable smoke test for TransformerNextClick
from models.transformer import TransformerNextClick
config = {'embedding_dim':32,'hidden_dim':64,'dropout':0.2,'attention_heads':4,'transformer_feedforward':64,'transformer_layers':2,'max_context':32}
model = TransformerNextClick(vocab_size=20, n_classes=7, config=config).eval()
x = torch.randint(0, 20, (4, 8)); lengths = torch.tensor([8, 6, 5, 3])
with torch.no_grad(): output = model(x, lengths)
print('transformer: output shape', tuple(output.shape))


## UNCP proposed hybrid

File: `experiments/models/uncp.py`  
Class: `UNCPNextClick`  
Runner name: `uncp`

In [1]:
from pathlib import Path
print(Path('/content/uncp_models/models/uncp.py').read_text())


Verified source file: /content/uncp_models/models/uncp.py
Class: UNCPNextClick
Runner name: uncp


```python
import torch
from torch import nn
from .common import sinusoidal_positions

class UNCPNextClick(nn.Module):
    """Proposed gated fusion of parallel GRU and pre-norm Transformer branches."""
    def __init__(self, vocab_size, n_classes, config):
        super().__init__()
        dim, hidden = config["embedding_dim"], config["hidden_dim"]
        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=0)
        self.gru = nn.GRU(dim, hidden, batch_first=True)
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=config["attention_heads"],
            dim_feedforward=config["transformer_feedforward"],
            dropout=config["dropout"], batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(
            layer, config["transformer_layers"], enable_nested_tensor=False)
        self.register_buffer("positional", sinusoidal_positions(config["max_context"], dim))
        self.transform_projection = nn.Linear(dim, hidden)
        self.gate = nn.Linear(2*hidden, hidden)
        self.norm = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(config["dropout"])
        self.output = nn.Linear(hidden, n_classes)

    def forward(self, x, lengths):
        x = x[:, :int(lengths.max())]
        batch = torch.arange(len(x))
        embedded = self.embedding(x)
        recurrent, _ = self.gru(embedded)
        recurrent = recurrent[batch, lengths-1]
        attended = self.transformer(
            embedded + self.positional[:x.shape[1]], src_key_padding_mask=x.eq(0))
        attended = self.transform_projection(attended[batch, lengths-1])
        gate = torch.sigmoid(self.gate(torch.cat([recurrent, attended], dim=1)))
        fused = self.norm(gate*recurrent + (1-gate)*attended)
        return self.output(self.dropout(fused))

```

In [ ]:
# Executable smoke test for UNCPNextClick
from models.uncp import UNCPNextClick
config = {'embedding_dim':32,'hidden_dim':64,'dropout':0.2,'attention_heads':4,'transformer_feedforward':64,'transformer_layers':2,'max_context':32}
model = UNCPNextClick(vocab_size=20, n_classes=7, config=config).eval()
x = torch.randint(0, 20, (4, 8)); lengths = torch.tensor([8, 6, 5, 3])
with torch.no_grad(): output = model(x, lengths)
print('uncp: output shape', tuple(output.shape))


## Transparent baselines

Uniform, popularity, repeat-last, prefix-frequency and Markov orders 1--4 are implemented separately from the trainable neural models.

In [1]:
print((ROOT / 'models' / 'baselines.py').read_text())


Verified source file: experiments/models/baselines.py
Baselines: uniform, popularity, repeat-last, prefix-frequency, Markov-1/2/3/4


```python
"""Transparent non-neural baselines fitted only on the training partition."""
import collections
import numpy as np

def baseline_rankings(train, test, encoder):
    labels, counts = encoder.labels, encoder.target_counts
    popular = sorted(labels, key=lambda label: (-counts[label], label))
    transition_counts = {
        order: collections.defaultdict(collections.Counter) for order in (1, 2, 3, 4)
    }
    for example in train:
        for order, mapping in transition_counts.items():
            if len(example["context"]) >= order:
                mapping[tuple(example["context"][-order:])][example["target"]] += 1

    def rank(counter):
        return sorted(labels, key=lambda label: (-counter[label], -counts[label], label))[:3]

    maps = {order: {context: rank(counter) for context, counter in mapping.items()}
            for order, mapping in transition_counts.items()}
    names = ("global_popularity", "repeat_last", "prefix_frequency", "markov1",
             "markov2_backoff", "markov3_backoff", "markov4_backoff")
    predictions = {name: [] for name in names}
    for example in test:
        last = example["context"][-1]
        backoff, markov = popular[:3], {}
        for order in (1, 2, 3, 4):
            if len(example["context"]) >= order:
                backoff = maps[order].get(tuple(example["context"][-order:]), backoff)
            markov[order] = backoff
        observed = collections.Counter(
            label for label in example["context"] if label in encoder.outputs)
        prefix = sorted(labels, key=lambda label: (-observed[label], -counts[label], label))[:3]
        predictions["global_popularity"].append(popular[:3])
        predictions["repeat_last"].append(
            ([last] if last in encoder.outputs else []) + [v for v in popular if v != last][:3])
        predictions["prefix_frequency"].append(prefix)
        for order in (1, 2, 3, 4):
            predictions["markov1" if order == 1 else f"markov{order}_backoff"].append(markov[order])
    result = {name: np.array([
        [example["target"] == ranked[0], example["target"] in ranked[:3]]
        for example, ranked in zip(test, values)], dtype=float)
        for name, values in predictions.items()}
    seen = np.array([example["target"] in encoder.outputs for example in test], dtype=float)
    result["uniform_random_expected"] = seen[:, None] * np.array(
        [1/len(labels), min(3, len(labels))/len(labels)])[None, :]
    return result

```